### Code libraries to review before operating (in the following order)<br>
1. intro_to_agents.agents.agents CompositeAgent

### Imports

In [ ]:
# Imports
import os

from dotenv import load_dotenv

from intro_to_agents.rag.embedders import SentenceTransformerEmbedder
from intro_to_agents.rag.vector_databases import ChromaDBVectorDB

from intro_to_agents.agents.llms import OpenAILLM
from intro_to_agents.agents.agents import MultiAgent, ChromaAgent, SQLiteAgent, ExcelAgent

# ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Make sure you have a .env file with your OpenAI API key
    # ex: OPENAI_API_KEY = "abcdefg12345"
    # a .env file is a text file that contains environment variables
    # it is used to store sensitive information (like API keys) in a secure way
    # it is NOT included in the repository (you don't want to share your API key with others), so you will need to create it yourself
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

# Instantiate the LLM
model = "gpt-5-mini" # you can change the model to whatever you want (see from intro_to_agents.agents.llms import OPENAI_TOKEN_LIMITS)
llm = OpenAILLM(api_key=api_key, model=model)

In [ ]:
# Load to vector database
# @ You: Identify the path where your vector database that you previously loaded is located
dbpath = r"C:\Users\sandidgeh\Downloads"

# Instantiate the same embedder you used to load the vector database
embedder = SentenceTransformerEmbedder()

# Instantiate the vector database
vdb = ChromaDBVectorDB(dbpath = dbpath, 
                       embedder = embedder, 
                       distance_measure = "cosine")

# Connect to vector database
vdb.initialize_db()

# Attach to a collection
vdb.initialize_collection("TGNSI_Commerce_Letters")

# Instantiate the RAG Agent
rag_agent = ChromaAgent(llm = llm, 
                        vectordb = vdb)

# Build the Agent "Card"
    # Provide a description of the Agent as to what types of questions it can answer
        # So "This agent can answer questions about..." Your description
rag_desc = "what type of products were traded between TGNSI and various countries as well as the perceived quality and economic benefits to each nation"

    # Provide the kwargs for how you want the  Agent to perform
rag_kwargs = {"k": 5, 
              "max_distance": 0.75, 
              "show_citations": True}

In [ ]:
# @You: Locate your SQLite database file
db_url = "../data/TGNSI_Trade.db"

# Provide context about the database to the LLM
db_desc = """
The database contains data international trades between various countries and the Great Nation of Sandman Island.
In order to successfully join tables, you must join any of the trade tables on the sectors table using their shared column: sector_code.
Additionally, to join the trade tables to the population table, you must join on their shared year columns.
""".rstrip()

# Instantiate the SQLiteAgent
sql_agent = SQLiteAgent(llm = llm, 
                        database_url = db_url, 
                        db_desc = db_desc, 
                        include_detail = True)

# Build the Agent "Card"
    # Provide a description of the Agent as to what types of questions it can answer
        # So "This agent can answer questions about..." Your description
sql_desc = "the dollar and quantity amounts of goods traded (as well as the category of goods and their security-risk classification) between TGNSI and various countries, the population of TGNSI."

    # Provide the kwargs for how you want the Agent to perform
sql_kwargs = {"view_sql": True}


In [ ]:
# Build the MultiAgent
    # Provide the names of the Agents
agent_names = ["Rag Agent", "SQL Agent"]

    # Provide the Agents themselves
agents = [rag_agent, sql_agent]

    # Provide the descriptions of the Agents
agent_descriptions = [rag_desc, sql_desc]

    # Provide the kwargs for how you want the MultiAgent to perform
agent_query_kwargs = [rag_kwargs, sql_kwargs]

# Instantiate the MultiAgent
multi_agent = MultiAgent(llm = llm, 
                         agent_names = agent_names, 
                         agents = agents, 
                         agent_descriptions = agent_descriptions, 
                         agent_query_kwargs = agent_query_kwargs)

In [ ]:
# Query MultiAgent
print(multi_agent.query("How much money did TGNSI make from trading with Mexico between 2020 and 2022?", 
                        show_logic = True))

In [ ]:
# Query MultiAgent
print(multi_agent.query("What goods do Mexico and TGNSI trade?", show_logic = True))